In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.append('/fast/pmayilvahanan/post_training/dapo/verl/notebooks')
from utils import transform_r1_to_qwen_format
import os

In [23]:
base_folder = "/fast/pmayilvahanan/post_training/self_distilled_datasets_neurips/"
# folders = ['DeepSeek-R1-Distill-Qwen-7B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0',
#            'Qwen2.5-3B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0',
#            'Qwen2.5-7B_openai_math_n_8_bsz_512_epochs_10_kl_coef_0.0_step_0',
#            'openai_math']

folders = ['Qwen2.5-1.5B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0',
           'Qwen2.5-3B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0',
           'Qwen2.5-7B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0',
           'Qwen2.5-14B_openai_math_n_8_bsz_128_epochs_1_kl_coef_0.0_step_0',
           'openai_math']



folders = [
           'Qwen2.5-7B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0',
           'Qwen2.5-14B_openai_math_n_8_bsz_128_epochs_1_kl_coef_0.0_step_0',
           'openai_math']

In [24]:
datasets = {}
for folder in folders:
    datasets[folder] = pd.read_parquet(base_folder + folder + "/train.parquet")

openai_math_test = pd.read_parquet(base_folder + "openai_math/test.parquet")

In [11]:
# ## This is fixed we don't need to do this anymore
# # fix openai_math formats first 
# # Make a copy of the dataset and rename 'solution' to 'answer' in the extra_info column
# # Also add a new column 'original_idx' from extra_info['index']
# openai_math_df = datasets['openai_math'].copy()
# openai_math_df['extra_info'] = openai_math_df['extra_info'].apply(
#     lambda x: {**x, 'answer': x.pop('solution')} if 'solution' in x else x
# )

# openai_math_test['extra_info'] = openai_math_test['extra_info'].apply(
#     lambda x: {**x, 'answer': x.pop('solution')} if 'solution' in x else x
# )

# # Extract 'index' from extra_info and create a new column 'original_idx'
# openai_math_df['original_idx'] = openai_math_df['extra_info'].apply(
#     lambda x: x.get('index') if 'index' in x else None
# )

# openai_math_test['original_idx'] = openai_math_test['extra_info'].apply(
#     lambda x: x.get('index') if 'index' in x else None
# )

# datasets['openai_math'] = openai_math_df


# def format_prompt(prompt_array):
#     if isinstance(prompt_array, np.ndarray) and len(prompt_array) > 0 and isinstance(prompt_array[0], dict):
#         content = prompt_array[0].get('content', '')
#         role = prompt_array[0].get('role', 'user')
#         new_content = f"system\nYou are a helpful assistant.\nuser\n{content}\nassistant\n"
#         return np.array([{'content': new_content, 'role': role}], dtype=object)
#     return prompt_array
# openai_math_test['prompt'] = openai_math_test['prompt'].apply(format_prompt)
# datasets['openai_math']['prompt'] = datasets['openai_math']['prompt'].apply(format_prompt)

# # Display the first prompt to verify\n\
# print(openai_math_test['prompt'][0])
# print(datasets['openai_math']['prompt'][0])
# print(datasets['Qwen2.5-14B_openai_math_n_8_bsz_128_epochs_1_kl_coef_0.0_step_0']['prompt'][0])

# openai_math_test.to_parquet('/fast/pmayilvahanan/post_training/self_distilled_datasets_neurips/openai_math/test.parquet')
# datasets['openai_math'].to_parquet('/fast/pmayilvahanan/post_training/self_distilled_datasets_neurips/openai_math/train.parquet')

[{'content': "system\nYou are a helpful assistant.\nuser\nConvert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$ Let's think step by step and output the final answer within \\boxed{}.\nassistant\n", 'role': 'user'}]
[{'content': "system\nYou are a helpful assistant.\nuser\nHow many vertical asymptotes does the graph of $y=\\frac{2}{x^2+x-6}$ have? Let's think step by step and output the final answer within \\boxed{}.\nassistant\n", 'role': 'user'}]
[{'content': "system\nYou are a helpful assistant.\nuser\nIf $a,b,c$ are integers from the set of positive integers less than $7$ such that \\begin{align*}\nabc&\\equiv 1\\pmod 7,\\\\\n5c&\\equiv 2\\pmod 7,\\\\\n6b&\\equiv 3+b\\pmod 7,\n\\end{align*}then what is the remainder when $a+b+c$ is divided by $7$? Let's think step by step and output the final answer within \\boxed{}.\nassistant\n", 'role': 'user'}]


In [ ]:
# # R1 distill models
# datasets['DeepSeek-R1-Distill-Qwen-7B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0']['prompt'][0]

# r1_dataset_key = 'DeepSeek-R1-Distill-Qwen-7B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0'
# datasets[r1_dataset_key] = datasets[r1_dataset_key].apply(transform_r1_to_qwen_format, axis=1)


In [25]:
# Get the keys for the datasets that are dataframes
dataframe_keys = [key for key in datasets.keys() if isinstance(datasets[key], pd.DataFrame)]

# Extract the original_idx values from each dataframe
original_indices = {}
for key in dataframe_keys:
    if 'original_idx' in datasets[key].columns:
        original_indices[key] = set(datasets[key]['original_idx'])

# Find common indices across all dataframes
if original_indices:
    common_indices = set.intersection(*original_indices.values())
    print(f"Found {len(common_indices)} common indices across {len(original_indices)} dataframes")
    
    # Create new dataframes with only the common indices
    filtered_datasets = {}
    for key in original_indices.keys():
        filtered_datasets[key] = datasets[key][datasets[key]['original_idx'].isin(common_indices)]
        print(f"Filtered {key}: {len(filtered_datasets[key])} rows")
else:
    print("No dataframes with 'original_idx' column found")

# Display the original dataset keys for reference
print("\nOriginal dataset keys:")
datasets.keys()

# Create random samples of the same size as common indices
random_sampled_datasets = {}
common_indices_count = len(common_indices) if original_indices else 0

if common_indices_count > 0:
    print(f"\nCreating random samples of size {common_indices_count} from each dataframe:")
    for key in original_indices.keys():
        # Sample randomly from the original dataset, same size as common indices
        if len(datasets[key]) >= common_indices_count:
            random_sampled_datasets[key] = datasets[key].sample(n=common_indices_count, random_state=42)
        else:
            # If dataset is smaller than common_indices_count, sample with replacement
            random_sampled_datasets[key] = datasets[key].sample(n=common_indices_count, replace=True, random_state=42)
        print(f"Random sampled {key}: {len(random_sampled_datasets[key])} rows")

Found 6424 common indices across 3 dataframes
Filtered Qwen2.5-7B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0: 6424 rows
Filtered Qwen2.5-14B_openai_math_n_8_bsz_128_epochs_1_kl_coef_0.0_step_0: 6424 rows
Filtered openai_math: 6424 rows

Original dataset keys:

Creating random samples of size 6424 from each dataframe:
Random sampled Qwen2.5-7B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0: 6424 rows
Random sampled Qwen2.5-14B_openai_math_n_8_bsz_128_epochs_1_kl_coef_0.0_step_0: 6424 rows
Random sampled openai_math: 6424 rows


In [16]:
# save filtered datasets
directory_path = '/fast/pmayilvahanan/post_training/self_distilled_datasets_neurips/'

# Save each filtered dataset to a file
for key in filtered_datasets.keys():
    output_path = os.path.join(directory_path, f"{key}/train_common_indices_7_14.parquet")
    filtered_datasets[key].to_parquet(output_path)
    print(f"Saved {key} dataset with {len(filtered_datasets[key])} rows to {output_path}")

 # Save random sampled datasets  
for key in random_sampled_datasets.keys():
    output_path = os.path.join(directory_path, f"{key}/train_random_sampled_7_14.parquet")
    random_sampled_datasets[key].to_parquet(output_path)
    print(f"Saved {key} dataset with {len(random_sampled_datasets[key])} rows to {output_path}")

Saved Qwen2.5-7B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0 dataset with 6424 rows to /fast/pmayilvahanan/post_training/self_distilled_datasets_neurips/Qwen2.5-7B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0/train_common_indices_7_14.parquet
Saved Qwen2.5-14B_openai_math_n_8_bsz_128_epochs_1_kl_coef_0.0_step_0 dataset with 6424 rows to /fast/pmayilvahanan/post_training/self_distilled_datasets_neurips/Qwen2.5-14B_openai_math_n_8_bsz_128_epochs_1_kl_coef_0.0_step_0/train_common_indices_7_14.parquet
Saved openai_math dataset with 6424 rows to /fast/pmayilvahanan/post_training/self_distilled_datasets_neurips/openai_math/train_common_indices_7_14.parquet
Saved Qwen2.5-7B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0 dataset with 6424 rows to /fast/pmayilvahanan/post_training/self_distilled_datasets_neurips/Qwen2.5-7B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0/train_random_sampled_7_14.parquet
Saved Qwen2.5-14B_openai_math_n_8_bsz_128_epochs_1_kl_coef_0.0_step_

In [58]:
filtered_datasets['Qwen2.5-3B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0']

,data_source,prompt,ability,extra_info,original_idx
1,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'To find the distance between the c...,9354
2,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,"{'answer': 'To find $g(x)$, we can rearrange t...",2813
3,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,"{'answer': 'To calculate $2 \nabla 5$, we will...",9112
4,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'Let \( r(x) \) denote the remainde...,9052
5,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'To solve \(\tan x = \sin x\) for \...,978
...,...,...,...,...,...
7088,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'Let's solve this step-by-step. 1....,5795
7089,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,"{'answer': 'Okay, let's break this down step b...",2093
7090,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'lahoma solved the quadratic equati...,9589
7091,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'To find the equation of the line i...,5467


In [61]:
# Calculate the average length of all answers in the Qwen2.5-7B dataset
df = filtered_datasets['Qwen2.5-7B_openai_math_n_8_bsz_512_epochs_10_kl_coef_0.0_step_0']
df = filtered_datasets['Qwen2.5-3B_openai_math_n_8_bsz_512_epochs_1_kl_coef_0.0_step_0']

# Extract the answer from each prompt (assuming the answer is in the assistant's response)
answer_lengths = []
for extra_info in df['extra_info']:
    answer_lengths.append(len(extra_info.get('answer', '')))

# Calculate and display the average answer length
avg_answer_length = sum(answer_lengths) / len(answer_lengths) if answer_lengths else 0
print(f"Average answer length: {avg_answer_length:.2f} characters")

# Display the dataframe as well
filtered_datasets['Qwen2.5-7B_openai_math_n_8_bsz_512_epochs_10_kl_coef_0.0_step_0']

Average answer length: 1155.00 characters


,data_source,prompt,ability,extra_info,original_idx
1,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'Given the quadratic functions \( f...,1943
3,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'To find the mid-point of a line se...,5526
5,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'To find the first nonzero digit to...,9593
6,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'Let's evaluate the given expressio...,8517
7,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'To find the number of positive two...,10557
...,...,...,...,...,...
8528,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'To compute \(\frac{1990^3 - 1000^3...,7233
8529,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'To determine how many meters the s...,6157
8530,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'To find the value of \(1 - 0.\over...,6833
8532,simplescaling/openaimath,[{'content': 'system You are a helpful assista...,math,{'answer': 'To find the least positive integer...,4887


In [ ]:
idx=7946
print(filtered_datasets[r1_dataset_key][filtered_datasets[r1_dataset_key]['original_idx'] == idx]['prompt'].iloc[0])
print(filtered_datasets[r1_dataset_key][filtered_datasets[r1_dataset_key]['original_idx'] == idx]['extra_info'].iloc[0])